**collect restaurant info on infatuation website**

In [2]:
!pip install requests beautifulsoup4 pymongo lxml

zsh:1: command not found: pip


In [3]:
!pip install python-dotenv

zsh:1: command not found: pip


In [5]:
import time
import random
import re
import json
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

/Users/anitacheng/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
# Business Context: These are foundational settings and utility functions for efficiently and politely scraping The Infatuation website.
# They ensure we can access the website's content and process text consistently.

# The base URL for The Infatuation website.
BASE = "https://www.theinfatuation.com"
# The specific index page for San Francisco neighborhoods, which will be our starting point for finding restaurant guides.
INDEX_URL = f"{BASE}/san-francisco/neighborhoods/san-francisco"

# Standard HTTP headers to mimic a web browser. This helps avoid being blocked by the website's servers.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

# Initialize a requests session. This allows for persistent parameters (like headers) across multiple requests
# and can improve performance by reusing TCP connections.
session = requests.Session()
session.headers.update(HEADERS)

# Function: polite_get
# Purpose: To make HTTP GET requests with a randomized delay to avoid overwhelming the server
# and to mimic human browsing behavior, preventing IP bans or rate limiting.
def polite_get(url, min_sleep=1.0, max_sleep=2.0):
    # Introduce a random delay to be polite to the server.
    time.sleep(random.uniform(min_sleep, max_sleep))
    # Send a GET request to the specified URL with a 30-second timeout.
    resp = session.get(url, timeout=30)
    # Raise an HTTPError for bad responses (4xx or 5xx client/server errors).
    resp.raise_for_status()
    return resp

# Function: normalize_text
# Purpose: Cleans up extracted text by removing extra whitespace.
# This is crucial for consistent data quality, especially for restaurant names, addresses, and descriptions.
def normalize_text(x):
    if not x:
        return None
    # Replace multiple whitespace characters with a single space, then strip leading/trailing whitespace.
    x = re.sub(r"\s+", " ", x).strip()
    # Return the cleaned text, or None if it becomes an empty string after cleaning.
    return x or None

# Function: is_rating
# Purpose: Checks if a given text string represents a numerical rating (e.g., "8.5" or "9").
# This helps in accurately parsing the restaurant rating information.
def is_rating(text):
    if not text:
        return False
    text = text.strip()
    # Regular expression to match a digit, optionally followed by a decimal point and another digit.
    return bool(re.fullmatch(r"\d(?:\.\d)?", text))

# Function: get_neighborhood_guide_urls
# Purpose: Scrapes the San Francisco neighborhood index page to find links to all restaurant guides.
# Business Context: These guide URLs are the entry points to discover Infatuation-featured restaurants,
# forming the basis for our 'editorial media coverage' dataset.
def get_neighborhood_guide_urls():
    # Fetch the HTML content of the San Francisco index page.
    html = polite_get(INDEX_URL).text
    # Parse the HTML content using BeautifulSoup for easy navigation.
    soup = BeautifulSoup(html, "html.parser")

    urls = set() # Use a set to store URLs to automatically handle duplicates.

    # Find all <a> (anchor) tags that have an 'href' attribute.
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        # Filter for links that are specifically restaurant guides.
        if "/guides/" not in href:
            continue

        # Construct the full absolute URL from the relative path.
        full = urljoin(BASE, href)

        # Ensure the guide is relevant to San Francisco, as some links might redirect elsewhere.
        if "/san-francisco/" not in full:
            continue

        urls.add(full) # Add the valid guide URL to our set.

    # Convert the set of unique URLs to a sorted list for consistent processing.
    return sorted(urls)

In [8]:
# Call the function to retrieve all neighborhood guide URLs for San Francisco.
guide_urls = get_neighborhood_guide_urls()
# Display the total number of guide URLs found and the first 10 URLs.
# This helps verify that the initial scraping for guide links was successful.
len(guide_urls), guide_urls[:10]

(11,
 ['https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-japantown-sf',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-the-sunset',
  'https://www.theinfatuation.com/san-francisco/guides/best-restaurants-near-fishermans-wharf-and-ghirardelli-square',
  'https://www.theinfatuation.com/san-francisco/guides/best-richmond-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/embarcadero-ferry-building-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/fidi-lunch-restaurants-sf',
  'https://www.theinfatuation.com/san-francisco/guides/noe-valley-restaurants',
  'https://www.theinfatuation.com/san-francisco/guides/north-beach-restaurants-san-francisco',
  'https://www.theinfatuation.com/san-francisco/guides/russian-hill-restaurants'])

In [9]:
# Function: extract_restaurant_cards_from_guide
# Purpose: To parse a single Infatuation guide URL and extract details for all listed restaurants.
# Business Context: Each extracted 'restaurant card' represents a featured restaurant,
# providing us with granular data points (name, rating, cuisine, neighborhood, description) that
# will be cross-referenced with survival rates.
def extract_restaurant_cards_from_guide(url):
    # Fetch the HTML content of the specific guide URL.
    html = polite_get(url).text
    # Parse the HTML content using BeautifulSoup for easier data extraction.
    soup = BeautifulSoup(html, "html.parser")

    restaurants = []  # List to store the structured data for each restaurant.
    seen = set()  # Set to track unique (name, review_url) combinations to prevent duplicate entries.

    # Iterate through all <a> (anchor) tags with an 'href' attribute on the page.
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        # Filter for links that lead to individual restaurant reviews.
        if "/reviews/" not in href:
            continue

        # Extract the restaurant's name from the link text and normalize it.
        name = normalize_text(a.get_text(" ", strip=True))
        if not name:
            continue # Skip if no valid name is found.

        # Construct the full absolute URL for the restaurant's review page.
        review_url = urljoin(BASE, href)

        # Navigate up the DOM tree from the link to find the main 'container' element
        # that holds all the details for this specific restaurant card.
        container = a
        for _ in range(6): # Check up to 6 parent levels.
            container = container.parent
            if container is None:
                break

            # Extract a blob of text and check for the presence of specific links within the container.
            # These links (maps, perfect-for, cuisine, neighborhood) indicate a complete restaurant card.
            text_blob = normalize_text(container.get_text(" ", strip=True)) or ""
            has_maps = bool(container.find("a", href=lambda x: x and "google.com/maps" in x))
            has_perfect_for = bool(container.find("a", href=lambda x: x and "/perfect-for/" in x))
            has_cuisine = bool(container.find("a", href=lambda x: x and "/cuisines/" in x))
            has_neighborhood = bool(container.find("a", href=lambda x: x and "/neighborhoods/" in x))

            # A valid container should have a substantial amount of text AND at least one of the key identifying links.
            if len(text_blob) > 80 and (has_maps or has_perfect_for or has_cuisine or has_neighborhood):
                break # Found the right container, exit loop.

        if container is None:
            continue # If no suitable container was found, skip this link.

        # Create a unique key (name, review_url) to identify and prevent adding duplicate restaurants.
        key = (name, review_url)
        if key in seen:
            continue # Skip if this restaurant has already been processed.
        seen.add(key)

        # Extract Address: Find the Google Maps link and get its text content.
        address = None
        maps_link = container.find("a", href=lambda x: x and "google.com/maps" in x)
        if maps_link:
            address = normalize_text(maps_link.get_text(" ", strip=True))

        # Extract Cuisines: Find all links pointing to cuisine categories.
        cuisines = []
        for c in container.find_all("a", href=lambda x: x and "/cuisines/" in x):
            txt = normalize_text(c.get_text(" ", strip=True))
            if txt and txt not in cuisines:
                cuisines.append(txt)

        # Extract Neighborhoods: Find all links pointing to neighborhood categories.
        neighborhoods = []
        for n in container.find_all("a", href=lambda x: x and "/neighborhoods/" in x):
            txt = normalize_text(n.get_text(" ", strip=True))
            if txt and txt not in neighborhoods:
                neighborhoods.append(txt)

        # Extract 'Perfect For' categories: Find all links pointing to occasion/type categories.
        perfect_for = []
        for p in container.find_all("a", href=lambda x: x and "/perfect-for/" in x):
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt and txt not in perfect_for:
                perfect_for.append(txt)

        # Extract Rating: Look for a numerical rating (e.g., 8.5) within the container's text.
        rating = None
        for s in container.stripped_strings:
            s = s.strip()
            if is_rating(s):
                rating = float(s)
                break

        # Prepare for Description Extraction: Get all distinct text strings within the container.
        texts = [normalize_text(t) for t in container.stripped_strings]
        texts = [t for t in texts if t] # Filter out empty strings.

        # Define elements to exclude from the description to avoid redundancy (data already captured).
        excluded = {name}
        if address:
            excluded.add(address)
        excluded.update(cuisines)
        excluded.update(neighborhoods)
        excluded.update(perfect_for)
        if rating is not None:
            # Add rating as string (e.g., "8.5") to exclusions, handling trailing zeros/decimals.
            excluded.add(str(rating).rstrip("0").rstrip("."))

        # Extract Description: Find the first sufficiently long text string that isn't part of the other extracted fields.
        description = None
        for t in texts:
            if t in excluded:
                continue # Skip if this text is an already-captured field.
            if len(t) >= 30: # Consider strings 30 characters or longer as potential descriptions.
                description = t
                break # Found description, move on.

        # Append all extracted data for the current restaurant to the list.
        restaurants.append({
            "name": name,
            "review_url": review_url,
            "guide_url": url,
            "address": address,
            "cuisines": cuisines,
            "neighborhoods": neighborhoods,
            "perfect_for": perfect_for,
            "rating": rating,
            "description": description,
        })

    return restaurants # Return the complete list of restaurants from this guide.

In [10]:
# Select the first guide URL from the `guide_urls` list for testing purposes.
test_url = guide_urls[0]
# Run the extraction function on this test URL to get a sample of restaurant data.
sample = extract_restaurant_cards_from_guide(test_url)
# Print the number of restaurants extracted from the sample guide to confirm data is being retrieved.
len(sample)

54

In [11]:
# Display the first 3 extracted restaurant entries from the `sample` list.
# This allows for a quick visual check of the data format and content.
sample[:3]

[{'name': 'San Ho Won',
  'review_url': 'https://www.theinfatuation.com/san-francisco/reviews/san-ho-won',
  'guide_url': 'https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf',
  'address': '2170 Bryant St, San Francisco, CA 94110',
  'cuisines': ['Korean'],
  'neighborhoods': ['Mission'],
  'perfect_for': ['Birthdays', 'Eating At The Bar', 'Special Occasions'],
  'rating': None,
  'description': None},
 {'name': 'READ THE REVIEW',
  'review_url': 'https://www.theinfatuation.com/san-francisco/reviews/san-ho-won',
  'guide_url': 'https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf',
  'address': '2170 Bryant St, San Francisco, CA 94110',
  'cuisines': ['Korean'],
  'neighborhoods': ['Mission'],
  'perfect_for': ['Birthdays', 'Eating At The Bar', 'Special Occasions'],
  'rating': 9.5,
  'description': 'San Ho Won’s galbi is capable of inducing epiphanies. It’s glistening, charred around the edges, and every bite of the melty meat co

In [12]:
# Initialize an empty list to accumulate all restaurant documents from all guides.
all_docs = []

# Loop through each guide URL identified in the initial scraping step.
for guide_url in guide_urls:
    try:
        # Call the extraction function for the current guide URL.
        docs = extract_restaurant_cards_from_guide(guide_url)
        # Print a status message indicating the guide processed and the number of restaurants found.
        print(f"{guide_url} -> {len(docs)} restaurants")
        # Add the extracted restaurants from the current guide to our master list.
        all_docs.extend(docs)
    except Exception as e:
        # If any error occurs during the scraping of a specific guide, print an error message.
        # This helps in identifying and debugging issues with problematic URLs.
        print(f"FAILED: {guide_url} -> {e}")

https://www.theinfatuation.com/san-francisco/guides/best-mission-restaurants-sf -> 54 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-japantown-sf -> 32 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-in-the-sunset -> 49 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-restaurants-near-fishermans-wharf-and-ghirardelli-square -> 24 restaurants
https://www.theinfatuation.com/san-francisco/guides/best-richmond-restaurants-sf -> 53 restaurants
https://www.theinfatuation.com/san-francisco/guides/embarcadero-ferry-building-restaurants-sf -> 47 restaurants
https://www.theinfatuation.com/san-francisco/guides/fidi-lunch-restaurants-sf -> 49 restaurants
https://www.theinfatuation.com/san-francisco/guides/noe-valley-restaurants -> 32 restaurants
https://www.theinfatuation.com/san-francisco/guides/north-beach-restaurants-san-francisco -> 42 restaurants
https://www.theinfatuation.com/san-francisco/guides/russia

In [13]:
# ------------------------------------------------------------
# Post-processing additions after Cell 8:
# 1) Deduplicate by review_url
# 2) Add normalized_name (same logic as Notebook 1)
# 3) Save to JSON
# 4) Print summary stats
# ------------------------------------------------------------

# Same normalization logic used for cross-source matching in MongoDB.
def normalize_name(name):
    if not name:
        return None
    name = name.lower()
    # Remove punctuation / non-alphanumeric characters.
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    # Strip common business suffixes.
    name = re.sub(r"\b(?:inc|llc|ltd|co|corp|corporation|company)\b", " ", name)
    # Collapse repeated whitespace.
    name = re.sub(r"\s+", " ", name).strip()
    return name or None

# 1) Deduplicate by unique review_url, keeping the first occurrence.
deduped_docs = []
seen_review_urls = set()

for doc in all_docs:
    review_url = doc.get("review_url")
    if not review_url or review_url in seen_review_urls:
        continue
    seen_review_urls.add(review_url)
    deduped_docs.append(doc)

# 2) Add normalized_name to each deduplicated record.
for doc in deduped_docs:
    doc["normalized_name"] = normalize_name(doc.get("name"))

# 3) Save the deduplicated + enriched records to JSON.
with open("infatuation_restaurants.json", "w", encoding="utf-8") as f:
    json.dump(deduped_docs, f, ensure_ascii=False, indent=2)

# 4) Print summary stats for verification.
total_scraped = len(all_docs)
after_dedup = len(deduped_docs)
with_ratings = sum(1 for doc in deduped_docs if doc.get("rating") is not None)
with_addresses = sum(1 for doc in deduped_docs if doc.get("address"))
unique_neighborhoods = sorted({
    neighborhood
    for doc in deduped_docs
    for neighborhood in (doc.get("neighborhoods") or [])
    if neighborhood
})

print(f"Total restaurants scraped across all guides: {total_scraped}")
print(f"Restaurants after deduplication: {after_dedup}")
print(f"Restaurants with ratings: {with_ratings}")
print(f"Restaurants with addresses: {with_addresses}")
print(f"Unique neighborhoods represented: {len(unique_neighborhoods)}")
print("Saved file: infatuation_restaurants.json")


Total restaurants scraped across all guides: 455
Restaurants after deduplication: 236
Restaurants with ratings: 22
Restaurants with addresses: 236
Unique neighborhoods represented: 13
Saved file: infatuation_restaurants.json


**collect registered restaurants**

In [14]:
import requests
import json
import re
from datetime import datetime
from collections import Counter, defaultdict

In [15]:
# ------------------------------------------------------------
# Step 1: Define the API request
# ------------------------------------------------------------

BASE_URL = "https://data.sfgov.org/resource/g8m3-pdis.json"

SELECT_FIELDS = [
    "dba_name",
    "full_business_address",
    "business_zip",
    "neighborhoods_analysis_boundaries",
    "location_start_date",
    "location_end_date",
    "naic_code",
    "naic_code_description",
    "business_corridor"
]

LIMIT = 1000

params_template = {
    "$select": ",".join(SELECT_FIELDS),
    "$where": "naic_code like '722%'",
    "$limit": LIMIT
}

In [16]:
# ------------------------------------------------------------
# Step 2: Paginate through all API results
# ------------------------------------------------------------

all_records = []
offset = 0

while True:
    params = params_template.copy()
    params["$offset"] = offset

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    batch = response.json()

    print(f"Fetched {len(batch)} rows at offset {offset}")

    all_records.extend(batch)

    if len(batch) < LIMIT:
        break

    offset += LIMIT

print(f"\nTotal records fetched: {len(all_records)}")


Fetched 1000 rows at offset 0
Fetched 1000 rows at offset 1000
Fetched 1000 rows at offset 2000
Fetched 1000 rows at offset 3000
Fetched 1000 rows at offset 4000
Fetched 1000 rows at offset 5000
Fetched 1000 rows at offset 6000
Fetched 1000 rows at offset 7000
Fetched 1000 rows at offset 8000
Fetched 1000 rows at offset 9000
Fetched 1000 rows at offset 10000
Fetched 1000 rows at offset 11000
Fetched 1000 rows at offset 12000
Fetched 1000 rows at offset 13000
Fetched 1000 rows at offset 14000
Fetched 1000 rows at offset 15000
Fetched 1000 rows at offset 16000
Fetched 150 rows at offset 17000

Total records fetched: 17150


In [22]:
# ------------------------------------------------------------
# Step 3: Helper functions for cleaning
# ------------------------------------------------------------

def parse_iso_date(date_str):
    if not date_str:
        return None
    try:
        return datetime.fromisoformat(date_str.split("T")[0])
    except Exception:
        return None

def compute_lifespan_days(start_dt, end_dt=None):
    if start_dt is None:
        return None
    if end_dt is None:
        end_dt = datetime.now()
    return (end_dt - start_dt).days


In [17]:
# ------------------------------------------------------------
# Step 4: Normalize restaurant names using regex
# ------------------------------------------------------------

def normalize_name(name):
    if not name:
        return None

    name = name.lower()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\b(inc|llc|corp|co|company|restaurant|restaurants)\b", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name if name else None


In [23]:
# ------------------------------------------------------------
# Step 5: Build cleaned records
# ------------------------------------------------------------

ingested_at = datetime.now().isoformat()

cleaned_records = []

for record in all_records:
    start_dt = parse_iso_date(record.get("location_start_date"))
    end_dt = parse_iso_date(record.get("location_end_date"))

    status = "Active" if not record.get("location_end_date") else "Closed"
    lifespan_days = compute_lifespan_days(start_dt, end_dt)

    cleaned_record = {
        "dba_name": record.get("dba_name"),
        "normalized_name": normalize_name(record.get("dba_name")),
        "full_business_address": record.get("full_business_address"),
        "business_zip": record.get("business_zip"),
        "neighborhoods_analysis_boundaries": record.get("neighborhoods_analysis_boundaries"),
        "location_start_date": record.get("location_start_date"),
        "location_end_date": record.get("location_end_date"),
        "status": status,
        "lifespan_days": lifespan_days,
        "naic_code": record.get("naic_code"),
        "naic_code_description": record.get("naic_code_description"),
        "business_corridor": record.get("business_corridor"),
        "ingested_at": ingested_at
    }

    cleaned_records.append(cleaned_record)

print(f"Cleaned records created: {len(cleaned_records)}")

Cleaned records created: 17150


In [26]:

COVID_CUTOFF = datetime(2020, 1, 1)

post_covid_records = []

for record in cleaned_records:
    end_dt = parse_iso_date(record.get("location_end_date"))

    # Keep:
    # - still active (no end date)
    # - or closed on/after 2020-01-01
    if end_dt is None or end_dt >= COVID_CUTOFF:
        post_covid_records.append(record)

print(f"Post-COVID analysis cohort size: {len(post_covid_records)}")
print(f"Cutoff date used: {COVID_CUTOFF.date()}")

post_covid_output_file = "sf_restaurants_post_covid.json"

with open(post_covid_output_file, "w", encoding="utf-8") as f:
    json.dump(post_covid_records, f, ensure_ascii=False, indent=2)

print(f"Saved post-COVID cohort to {post_covid_output_file}")

Post-COVID analysis cohort size: 14459
Cutoff date used: 2020-01-01
Saved post-COVID cohort to sf_restaurants_post_covid.json


In [ ]:
# ------------------------------------------------------------
# Step 6: Save the cleaned full dataset as JSON
# ------------------------------------------------------------

post_covid_output_file = "sf_restaurants_post_2020.json"

with open(post_covid_output_file, "w", encoding="utf-8") as f:
    json.dump(post_covid_records, f, ensure_ascii=False, indent=2)

print(f"Saved post-2020 cohort to {post_covid_output_file}")


Saved post-2020 cohort to sf_restaurants_post_2020.json


**join and save on MongoDB**

In [ ]:
pip install 

In [29]:
import json
import re
from typing import List, Dict, Any, Optional
from pymongo import MongoClient, UpdateOne

# =========================
# 1) CONFIG
# =========================
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "sf_restaurants_project"

SF_JSON_PATH = "sf_restaurants_post_2020.json"          # output from SF notebook
INFATUATION_JSON_PATH = "infatuation_restaurants.json"  # output from Infatuation notebook

SF_COLLECTION = "sf_restaurants"
INF_COLLECTION = "infatuation_restaurants"
JOINED_COLLECTION = "restaurants_joined"

# =========================
# 2) HELPERS
# =========================
STOPWORDS = {
    "restaurant", "cafe", "bar", "grill", "kitchen", "sf",
    "san", "francisco", "the", "&"
}

def load_json_file(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        return [data]
    else:
        raise ValueError(f"Unexpected JSON format in {path}")

def normalize_text(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def normalize_name(name: Optional[str]) -> Optional[str]:
    s = normalize_text(name)
    if not s:
        return None

    tokens = [t for t in s.split() if t not in STOPWORDS]
    s = " ".join(tokens)

    # remove common business suffixes
    s = re.sub(r"\b(llc|inc|co|corp|corporation|ltd)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def normalize_address(addr: Optional[str]) -> Optional[str]:
    s = normalize_text(addr)
    if not s:
        return None

    # standardize common street suffixes
    replacements = {
        r"\bstreet\b": "st",
        r"\bavenue\b": "ave",
        r"\bboulevard\b": "blvd",
        r"\broad\b": "rd",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bplace\b": "pl",
        r"\bcourt\b": "ct",
        r"\bterrace\b": "ter",
        r"\bhighway\b": "hwy",
        r"\bcalifornia\b": "ca",
    }
    for pattern, repl in replacements.items():
        s = re.sub(pattern, repl, s)

    # keep only SF-relevant part if needed
    s = s.replace("san francisco ca", "san francisco ca")
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def first_non_null(*values):
    for v in values:
        if v not in (None, "", [], {}):
            return v
    return None

# =========================
# 3) PREP RECORDS
# =========================
def prepare_sf_docs(sf_docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    prepared = []

    for doc in sf_docs:
        raw_name = doc.get("dba_name")
        raw_address = doc.get("full_business_address")

        doc["source"] = "sf_open_data"
        doc["join_name"] = first_non_null(doc.get("normalized_name"), normalize_name(raw_name))
        doc["join_address"] = normalize_address(raw_address)

        prepared.append(doc)

    return prepared

def prepare_infatuation_docs(inf_docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    prepared = []

    for doc in inf_docs:
        raw_name = doc.get("name")
        raw_address = doc.get("address")

        # remove noisy pseudo-records
        if raw_name and raw_name.strip().upper() in {"READ THE REVIEW", "BOOK NOW"}:
            continue

        doc["source"] = "infatuation"
        doc["join_name"] = normalize_name(raw_name)
        doc["join_address"] = normalize_address(raw_address)

        prepared.append(doc)

    return prepared

# =========================
# 4) JOIN LOGIC
# =========================
def build_inf_lookup(inf_docs: List[Dict[str, Any]]):
    by_address = {}
    by_name = {}

    for doc in inf_docs:
        if doc.get("join_address"):
            by_address.setdefault(doc["join_address"], []).append(doc)
        if doc.get("join_name"):
            by_name.setdefault(doc["join_name"], []).append(doc)

    return by_address, by_name

def choose_best_match(sf_doc: Dict[str, Any], candidates: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    if not candidates:
        return None

    # prefer exact address + exact name if possible
    for c in candidates:
        if (
            sf_doc.get("join_address") == c.get("join_address")
            and sf_doc.get("join_name") == c.get("join_name")
        ):
            return c

    # otherwise just take first
    return candidates[0]

def join_records(sf_docs: List[Dict[str, Any]], inf_docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    by_address, by_name = build_inf_lookup(inf_docs)
    joined = []

    for sf_doc in sf_docs:
        match_method = None
        inf_match = None

        # 1) address match first
        if sf_doc.get("join_address") and sf_doc["join_address"] in by_address:
            inf_match = choose_best_match(sf_doc, by_address[sf_doc["join_address"]])
            match_method = "address"

        # 2) fallback to name match
        elif sf_doc.get("join_name") and sf_doc["join_name"] in by_name:
            inf_match = choose_best_match(sf_doc, by_name[sf_doc["join_name"]])
            match_method = "name"

        merged = {
            "join_keys": {
                "join_name": sf_doc.get("join_name"),
                "join_address": sf_doc.get("join_address"),
            },
            "match_found": inf_match is not None,
            "match_method": match_method,
            "sf": sf_doc,
            "infatuation": inf_match
        }

        # optional flattened fields for easier querying
        merged["restaurant_name"] = first_non_null(
            sf_doc.get("dba_name"),
            inf_match.get("name") if inf_match else None
        )
        merged["address"] = first_non_null(
            sf_doc.get("full_business_address"),
            inf_match.get("address") if inf_match else None
        )
        merged["sf_neighborhood"] = sf_doc.get("neighborhoods_analysis_boundaries")
        merged["infatuation_neighborhoods"] = inf_match.get("neighborhoods") if inf_match else None
        merged["status"] = sf_doc.get("status")
        merged["lifespan_days"] = sf_doc.get("lifespan_days")
        merged["rating"] = inf_match.get("rating") if inf_match else None
        merged["cuisines"] = inf_match.get("cuisines") if inf_match else None
        merged["perfect_for"] = inf_match.get("perfect_for") if inf_match else None
        merged["review_url"] = inf_match.get("review_url") if inf_match else None
        merged["guide_url"] = inf_match.get("guide_url") if inf_match else None

        joined.append(merged)

    return joined

# =========================
# 5) SAVE TO MONGODB
# =========================
def upsert_many(collection, docs: List[Dict[str, Any]], unique_key: str):
    ops = []

    for doc in docs:
        key_val = doc.get(unique_key)
        if not key_val:
            continue

        ops.append(
            UpdateOne(
                {unique_key: key_val},
                {"$set": doc},
                upsert=True
            )
        )

    if ops:
        result = collection.bulk_write(ops)
        print(f"{collection.name}: upserted/updated {len(ops)} docs")
        return result
    else:
        print(f"{collection.name}: no valid docs to write")
        return None

def main():
    client = MongoClient(MONGO_URI)
    db = client[DB_NAME]

    sf_raw = load_json_file(SF_JSON_PATH)
    inf_raw = load_json_file(INFATUATION_JSON_PATH)

    sf_docs = prepare_sf_docs(sf_raw)
    inf_docs = prepare_infatuation_docs(inf_raw)
    joined_docs = join_records(sf_docs, inf_docs)

    # optional: clear old collections
    db[SF_COLLECTION].delete_many({})
    db[INF_COLLECTION].delete_many({})
    db[JOINED_COLLECTION].delete_many({})

    # raw collections
    if sf_docs:
        db[SF_COLLECTION].insert_many(sf_docs)
    if inf_docs:
        db[INF_COLLECTION].insert_many(inf_docs)
    if joined_docs:
        db[JOINED_COLLECTION].insert_many(joined_docs)

    # helpful indexes
    db[SF_COLLECTION].create_index("join_name")
    db[SF_COLLECTION].create_index("join_address")

    db[INF_COLLECTION].create_index("join_name")
    db[INF_COLLECTION].create_index("join_address")

    db[JOINED_COLLECTION].create_index("match_found")
    db[JOINED_COLLECTION].create_index("match_method")
    db[JOINED_COLLECTION].create_index("join_keys.join_name")
    db[JOINED_COLLECTION].create_index("join_keys.join_address")
    db[JOINED_COLLECTION].create_index("sf_neighborhood")
    db[JOINED_COLLECTION].create_index("rating")

    total = len(joined_docs)
    matched = sum(1 for d in joined_docs if d["match_found"])
    print(f"Total SF records processed: {len(sf_docs)}")
    print(f"Total Infatuation records processed: {len(inf_docs)}")
    print(f"Total joined records: {total}")
    print(f"Matched records: {matched}")
    print(f"Unmatched records: {total - matched}")

if __name__ == "__main__":
    main()

Total SF records processed: 14459
Total Infatuation records processed: 236
Total joined records: 14459
Matched records: 223
Unmatched records: 14236
